In [ ]:
1

In [17]:
from nocode_robot_programming.state_decision.plots.benchmark_plot import visualize_accuracies, plot_heatmap
import numpy as np
import matplotlib.pyplot as plt
from typing import List
from pathlib import Path

def plot_heatmap(matrix: np.ndarray, x_labels: List[str], y_labels: List[str], title: str, save_path: Path, jupyter_plot: bool, line=True):
    fig = plt.figure(figsize=(max(6, len(x_labels)*0.8), max(3, len(y_labels)*0.35)))
    ax = fig.add_subplot(111)

    im = ax.imshow(matrix, aspect="auto")

    ax.set_xticks(np.arange(len(x_labels)))
    ax.set_yticks(np.arange(len(y_labels)))
    ax.set_xticklabels(x_labels, rotation=45, ha="right")
    ax.set_yticklabels(y_labels)

    ax.set_ylabel("Models")
    ax.set_title(title)

    # --- vertical separator between column groups ---
    if line:
        ax.axvline(2.5, color="black", linewidth=1.5)

    # --- group titles (positioned in axis coordinates) ---
    if line:
        ax.text(0.25, 1.00, "Tasks", transform=ax.transAxes,
                ha="center", va="bottom", fontsize=8, fontweight="bold")

        ax.text(0.75, 1.00, "Modality", transform=ax.transAxes,
                ha="center", va="bottom", fontsize=8, fontweight="bold")

    # annotate cells
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            ax.text(j, i, f"{matrix[i, j]:.1f}",
                    ha="center", va="center", fontsize=10)

    fig.tight_layout()
    fig.savefig(save_path, dpi=160, bbox_inches="tight")
    plt.close(fig)

# State Estimation

In [19]:
m = np.array([
    [75.1, 71.5, 84.5, 78.9, 74.6, 74.7],
    [76.2, 72.3, 87.8, 80.8, 77.0, 75.1],
    [82.6, 74.0, 88.6, 82.8, 77.7, 80.7],
    [77.4, 76.8, 84.2, 82.1, 78.3, 76.4],
    [71.8, 71.5, 85.4, 78.5, 76.2, 71.4],
    [69.5, 67.8, 73.3, 69.2, 68.0, 72.3],
    [64.0, 59.5, 68.7, 65.3, 63.0, 61.5],
])
plot_heatmap(m, ['Peg\npick', 'Probe\nmeasure', 'Cable\nwrap', '$gst$', '$joy$', '$kin$'],
             ['dinov2 small mean', 'dinov3 small mean', 'dinov2 small concat', 'dinov2 small attn', 'dinov2 small MIL', 'SIFT', 'AEGP Multiclass'],
             "Test Accuracy (%)", "state_estimation.pdf", False)

# Anomaly

In [7]:
m = np.array([
    [65.9, 58.5, 75.7, 65.5, 70.1, 60.5],
    [61.0, 58.6, 77.9, 61.8, 71.9, 60.5],
    [78.3, 66.2, 87.9, 69.2, 76.9, 81.3],
    [79.7, 67.1, 74.8, 75.1, 79.4, 63.6],
    [24.1, 25.6, 18.7, 31.1, 16.6, 21.5],
])
plot_heatmap(m, ['Peg\npick', 'Probe\nmeasure', 'Cable\nwrap', '$gst$', '$joy$', '$kin$'],
             ['dinov2 small mean', 'dinov3 small mean', 'dinov2 small concat', 'dinov2 small attn', 'dinov2 small MIL', ], #'SIFT', 'AEGP Multiclass'], 
             "Test Accuracy (%)", "test_save.pdf", False)

# Additional

In [20]:
m = np.array([
    [92.4, 93.7, 79.8, 66.5, 55.9, 55.1, 49.0],
    [75.7, 73.4, 64.2, 58.1, 48.7, 45.3, 38.0],
    [92.4, 95.2, 77.6, 67.0, 56.2, 48.5, 43.2],
    [92.4, 94.6, 95.6, 94.0, 87.4, 82.0, 73.1],
    [92.4, 68.1, 46.2, 41.4, 28.5, 30.6, 21.4],
    [92.9, 88.7, 66.8, 62.5, 54.9, 43.7, 40.0],
    [92.4, 42.1, 27.7, 26.0, 19.0, 22.5, 15.7]
])
plot_heatmap(m, ['2 classes', '3 classes', '4 classes', '5 classes', '6 classes', '7 classes', '8 classes'],
             ['dinov2 small mean', 'dinov3 small mean', 'dinov2 small concat', 'dinov2 small attn', 'dinov2 small MIL', 'SIFT', 'AEGP Multiclass'], 
             "Test Accuracy (%)", "test_save_additional.pdf", False, line=False)

# State Estimation with +1/+2 trajectories

In [ ]:
from nocode_robot_programming.state_decision.plots.benchmark_plot import visualize_accuracies, plot_heatmap
import numpy as np
import matplotlib.pyplot as plt
from typing import List
from pathlib import Path

def plot_heatmap(matrix: np.ndarray, x_labels: List[str], y_labels: List[str], title: str, save_path: Path, jupyter_plot: bool):
    """Plot a heatmap. matrix can be 2-D (n_models, n_tasks) or 3-D (n_models, n_tasks, 3).
    In the 3-D case each cell has a left-to-right gradient from v[0] to max(v) and shows
    'no-inj / 1-traj / 2-traj' values as text."""
    n_rows, n_cols = matrix.shape[:2]
    fig = plt.figure(figsize=(max(6, n_cols * 0.8), max(4.5, n_rows * (0.75 if matrix.ndim == 3 else 0.6))))
    ax = fig.add_subplot(111)

    if matrix.ndim == 3:
        # Build a high-resolution gradient image: each cell gets N horizontal pixels.
        N = 60
        vmin, vmax = matrix.min(), matrix.max()
        grad = np.zeros((n_rows, n_cols * N))
        for i in range(n_rows):
            for j in range(n_cols):
                v = matrix[i, j]
                grad[i, j * N:(j + 1) * N] = np.linspace(v[0], max(v), N)
        ax.imshow(grad, aspect="auto", vmin=vmin, vmax=vmax,
                  extent=[-0.5, n_cols - 0.5, n_rows - 0.5, -0.5])
        # cell separators
        for j in range(1, n_cols):
            ax.axvline(x=j - 0.5, color="white", linewidth=0.5, alpha=0.6)
        # text annotations
        for i in range(n_rows):
            for j in range(n_cols):
                v = matrix[i, j]
                brightness = (grad[i, j * N + N // 2] - vmin) / max(vmax - vmin, 1e-9)
                color = "white" if brightness < 0.55 else "black"
                ax.text(j, i, f"{v[0]:.1f} /\n {v[1]:.1f} /\n {v[2]:.1f}",
                        ha="center", va="center", fontsize=9, color=color)
    else:
        ax.imshow(matrix, aspect="auto")
        for i in range(n_rows):
            for j in range(n_cols):
                ax.text(j, i, f"{matrix[i, j]:.1f}", ha="center", va="center", fontsize=8)

    ax.set_xticks(np.arange(n_cols))
    ax.set_yticks(np.arange(n_rows))
    ax.set_xticklabels(x_labels, rotation=45, ha="right")
    ax.set_yticklabels(y_labels)
    ax.set_xlabel("Tasks")
    ax.set_ylabel("Models")
    ax.set_title(title)
    fig.tight_layout()

    if jupyter_plot:
        ipt.save()
    else:
        fig.savefig(save_path, dpi=160, bbox_inches="tight")
        plt.close(fig)

In [ ]:
import sys, importlib
sys.path.insert(0, '.')
import custom_plot; importlib.reload(custom_plot)
from custom_plot import plot_switcher_accuracy_split

m = np.array([
    [[82.1, 91.1, 94.0], [76.0, 78.0, 85.5], [86.2, 89.4, 96.3], [82.6, 86.5, 93.1], [81.1, 84.4, 90.6], [77.7, 82.8, 88.3]],
    [[89.3, 89.0, 89.7], [73.8, 75.4, 84.7], [90.2, 94.1, 97.7], [83.6, 85.4, 90.6], [82.0, 85.1, 92.5], [81.8, 82.3, 86.6]],
    [[80.3, 87.6, 89.3], [70.7, 74.4, 83.6], [85.8, 90.0, 95.4], [79.6, 83.5, 90.7], [76.8, 82.4, 89.4], [75.9, 80.9, 85.1]],
    [[81.8, 80.5, 82.8], [71.1, 75.4, 83.0], [89.4, 96.7, 97.7], [81.8, 85.1, 89.0], [79.4, 84.1, 90.2], [75.9, 79.4, 82.8]],
    [[68.6, 86.6, 88.9], [70.2, 74.5, 75.1], [92.5, 95.0, 97.2], [79.3, 86.6, 88.6], [75.1, 86.3, 87.3], [73.8, 77.9, 79.2]],
    [[71.8, 82.1, 84.6], [66.7, 67.8, 74.7], [73.1, 77.9, 85.3], [69.7, 73.9, 81.5], [67.7, 77.4, 82.8], [71.9, 72.5, 77.0]],
    [[64.0, 64.0, 64.0], [59.5, 59.5, 59.5], [68.7, 68.7, 68.7], [65.3, 65.3, 65.3], [63.0, 63.0, 63.0], [61.5, 61.5, 61.5]],
])

model_labels = [
    'dinov2 small attn', 'dinov2 small concat', 'dinov2 small mean',
    'dinov3 small mean', 'dinov2 small MIL', 'SIFT', 'AEGP Multiclass',
]
model_aliases = ['DINOv2 attn', 'DINOv2 concat', 'DINOv2 mean', 'DINOv3 mean', 'DINOv2 MIL', 'SIFT', 'AEGP']

plot_switcher_accuracy_split(
    matrix=m,
    model_labels=model_labels,
    model_aliases=model_aliases,
    save_dir='.',
    prefix='state_estimation',
    jupyter_plot=True,
)

# State estimation with +1/+2 trajectories as a plot

In [ ]:
from pathlib import Path
from typing import List, Sequence, Optional, Tuple

import numpy as np
import matplotlib.pyplot as plt


COLORBLIND_SAFE_COLORS = [
    "#0072B2",  # blue
    "#D55E00",  # vermillion
    "#009E73",  # bluish green
    "#CC79A7",  # reddish purple
    "#E69F00",  # orange
    "#56B4E9",  # sky blue
    "#000000",  # black
]


MODEL_MARKERS = [
    "o",  # DINOv2 attn
    "s",  # DINOv2 concat
    "^",  # DINOv2 mean
    "D",  # DINOv3 mean
    "v",  # DINOv2 MIL
    "P",  # SIFT
    "X",  # AEGP
]


MODEL_LINESTYLES = [
    "-",   # DINOv2 attn
    "-",   # DINOv2 concat
    "-",   # DINOv2 mean
    "-",   # DINOv3 mean
    "-",   # DINOv2 MIL
    "--",  # SIFT
    ":",   # AEGP
]


def _non_overlapping_label_positions(
    target_y: np.ndarray,
    ymin: float,
    ymax: float,
    min_gap: float = 2.8,
    pad_frac: float = 0.03,
):
    """
    Return label y-positions sorted from highest target accuracy to lowest,
    while enforcing a minimum vertical gap between labels.
    """
    target_y = np.asarray(target_y, dtype=float)
    n = len(target_y)

    order = np.argsort(-target_y)
    y_sorted = target_y[order].copy()

    span = ymax - ymin
    top = ymax - pad_frac * span
    bottom = ymin + pad_frac * span

    y_sorted = np.clip(y_sorted, bottom, top)

    # Top-down pass: enforce descending label positions.
    for i in range(1, n):
        y_sorted[i] = min(y_sorted[i], y_sorted[i - 1] - min_gap)

    # If the label stack falls below the axis, shift it upward.
    if y_sorted[-1] < bottom:
        y_sorted += bottom - y_sorted[-1]

    # If shifting upward exceeds the top, compress the stack.
    if y_sorted[0] > top:
        if n == 1:
            y_sorted[0] = top
        else:
            available_gap = (top - bottom) / (n - 1)
            gap = min(min_gap, available_gap)
            y_sorted = top - gap * np.arange(n)

    return order, y_sorted


def _add_direct_model_labels(
    ax,
    line_handles,
    line_values_at_anchor: np.ndarray,
    model_names: Sequence[str],
    ylim: Tuple[float, float],
    label_x: float = 1.04,
    min_gap: float = 2.8,
    fontsize: int = 11,
):
    """
    Place model labels directly to the right of the right-most subplot.

    Labels are ranked by final accuracy. The best-performing model is placed
    highest, and the remaining labels are pushed downward while avoiding overlap.
    """
    ymin, ymax = ylim

    order, label_y_sorted = _non_overlapping_label_positions(
        line_values_at_anchor,
        ymin=ymin,
        ymax=ymax,
        min_gap=min_gap,
    )

    for rank, model_idx in enumerate(order):
        line = line_handles[model_idx]
        color = line.get_color()

        x_data = line.get_xdata()
        y_data = line.get_ydata()

        anchor_x = x_data[-1]
        anchor_y = y_data[-1]
        label_y = label_y_sorted[rank]

        ax.annotate(
            "",
            xy=(anchor_x, anchor_y),
            xycoords="data",
            xytext=(label_x - 0.012, label_y),
            textcoords=ax.get_yaxis_transform(),
            arrowprops=dict(
                arrowstyle="-",
                color=color,
                lw=1.0,
                alpha=0.85,
            ),
            annotation_clip=False,
        )

        ax.text(
            label_x,
            label_y,
            model_names[model_idx],
            transform=ax.get_yaxis_transform(),
            ha="left",
            va="center",
            fontsize=fontsize,
            color=color,
            clip_on=False,
            bbox=dict(
                facecolor="white",
                edgecolor="none",
                alpha=0.75,
                pad=0.8,
            ),
        )


def plot_switcher_accuracy_split_direct_labels(
    matrix: np.ndarray,
    model_labels: List[str],
    save_dir: Path | str = ".",
    prefix: str = "state_estimation",
    jupyter_plot: bool = False,
    ylim: Tuple[float, float] = (55.0, 100.0),
    train_labels: Sequence[str] = ("nominal", "+1", "+2 traj."),
    task_labels: Sequence[str] = ("Peg pick", "Probe measure", "Cable wrap"),
    modality_labels: Sequence[str] = ("gestures", "joystick", "kinesthetic"),
    model_aliases: Optional[Sequence[str]] = None,
    colors: Optional[Sequence[str]] = None,
    markers: Optional[Sequence[str]] = None,
    linestyles: Optional[Sequence[str]] = None,
    min_label_gap: float = 3.2,
    figsize_per_panel: Tuple[float, float] = (3.15, 3.25),
    base_fontsize: int = 11,
    label_fontsize: int = 11,
    title_fontsize: int = 12,
):
    """
    Plot Switcher accuracy as two compact figures with direct labels.

    Expected matrix shape:
        (n_models, 6, 3)

    Columns:
        0: Peg pick
        1: Probe measure
        2: Cable wrap
        3: gestures
        4: joystick
        5: kinesthetic

    Third dimension:
        0: nominal training set
        1: +1 trajectory
        2: +2 trajectories

    The generated files are:
        <prefix>_by_task.pdf
        <prefix>_by_modality.pdf
    """
    matrix = np.asarray(matrix, dtype=float)

    if matrix.ndim != 3:
        raise ValueError(f"Expected a 3-D matrix, got shape {matrix.shape}.")

    if matrix.shape[1] != 6:
        raise ValueError(
            f"Expected 6 columns: 3 tasks + 3 modalities, got {matrix.shape[1]}."
        )

    if matrix.shape[2] != 3:
        raise ValueError(
            f"Expected 3 training-set conditions, got {matrix.shape[2]}."
        )

    if matrix.shape[0] != len(model_labels):
        raise ValueError(
            f"Number of model labels ({len(model_labels)}) does not match "
            f"matrix rows ({matrix.shape[0]})."
        )

    if model_aliases is None:
        model_aliases = model_labels

    if len(model_aliases) != len(model_labels):
        raise ValueError("model_aliases must have the same length as model_labels.")

    if colors is None:
        colors = COLORBLIND_SAFE_COLORS

    if markers is None:
        markers = MODEL_MARKERS

    if linestyles is None:
        linestyles = MODEL_LINESTYLES

    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)

    def _make_plot(
        col_ids: Sequence[int],
        col_labels: Sequence[str],
        filename: str,
    ):
        x = np.arange(len(train_labels))

        fig, axes = plt.subplots(
            1,
            len(col_ids),
            figsize=(figsize_per_panel[0] * len(col_ids), figsize_per_panel[1]),
            sharey=True,
        )

        if len(col_ids) == 1:
            axes = [axes]

        rightmost_line_handles = None
        rightmost_anchor_values = None

        for ax_idx, (ax, col_id, col_label) in enumerate(zip(axes, col_ids, col_labels)):
            line_handles = []

            for model_idx, _ in enumerate(model_aliases):
                line_color = colors[model_idx % len(colors)]
                marker = markers[model_idx % len(markers)]
                linestyle = linestyles[model_idx % len(linestyles)]

                line, = ax.plot(
                    x,
                    matrix[model_idx, col_id, :],
                    marker=marker,
                    linestyle=linestyle,
                    color=line_color,
                    linewidth=2.1,
                    markersize=5.5,
                    markeredgewidth=0.8,
                )
                line_handles.append(line)

            ax.set_title(col_label, fontsize=title_fontsize)
            ax.set_xticks(x)
            ax.set_xticklabels(train_labels, fontsize=base_fontsize)
            ax.set_ylim(*ylim)
            ax.set_xlabel("Training set", fontsize=base_fontsize)
            ax.tick_params(axis="y", labelsize=base_fontsize)

            ax.grid(
                True,
                which="major",
                axis="both",
                linestyle="--",
                linewidth=0.6,
                alpha=0.45,
            )
            ax.set_axisbelow(True)

            for spine in ["top", "right"]:
                ax.spines[spine].set_visible(False)

            if ax_idx == len(col_ids) - 1:
                rightmost_line_handles = line_handles
                rightmost_anchor_values = matrix[:, col_id, -1]

        axes[0].set_ylabel("Test accuracy (%)", fontsize=base_fontsize)

        _add_direct_model_labels(
            ax=axes[-1],
            line_handles=rightmost_line_handles,
            line_values_at_anchor=rightmost_anchor_values,
            model_names=model_aliases,
            ylim=ylim,
            min_gap=min_label_gap,
            fontsize=label_fontsize,
        )

        # Leave right margin for direct labels.
        fig.tight_layout(rect=(0.0, 0.0, 0.78, 1.0))

        save_path = save_dir / filename
        fig.savefig(save_path, dpi=300, bbox_inches="tight")

        if jupyter_plot:
            plt.show()
        else:
            plt.close(fig)

        return save_path

    task_path = _make_plot(
        col_ids=[0, 1, 2],
        col_labels=task_labels,
        filename=f"{prefix}_by_task.pdf",
    )

    modality_path = _make_plot(
        col_ids=[3, 4, 5],
        col_labels=modality_labels,
        filename=f"{prefix}_by_modality.pdf",
    )

    return task_path, modality_path


m = np.array([ 
    [[82.1, 91.1, 94.0], [76.0, 78.0, 85.5], [86.2, 89.4, 96.3], [82.6, 86.5, 93.1], [81.1, 84.4, 90.6], [77.7, 82.8, 88.3]], 
    [[89.3, 89.0, 89.7], [73.8, 75.4, 84.7], [90.2, 94.1, 97.7], [83.6, 85.4, 90.6], [82.0, 85.1, 92.5], [81.8, 82.3, 86.6]], 
    [[80.3, 87.6, 89.3], [70.7, 74.4, 83.6], [85.8, 90.0, 95.4], [79.6, 83.5, 90.7], [76.8, 82.4, 89.4], [75.9, 80.9, 85.1]], 
    [[81.8, 80.5, 82.8], [71.1, 75.4, 83.0], [89.4, 96.7, 97.7], [81.8, 85.1, 89.0], [79.4, 84.1, 90.2], [75.9, 79.4, 82.8]], 
    [[68.6, 86.6, 88.9], [70.2, 74.5, 75.1], [92.5, 95.0, 97.2], [79.3, 86.6, 88.6], [75.1, 86.3, 87.3], [73.8, 77.9, 79.2]], 
    [[71.8, 82.1, 84.6], [66.7, 67.8, 74.7], [73.1, 77.9, 85.3], [69.7, 73.9, 81.5], [67.7, 77.4, 82.8], [71.9, 72.5, 77.0]], 
    [[61.9, 68.3, 72.8], [64.0, 62.4, 65.0 ], [76.5, 77.0, 87.2 ], [69.3, 70.0, 77.6 ], [67.9, 70.0, 74.9 ], [63.8, 61.1, 67.5]], 
])


model_labels = [
    "dinov2 small attn",
    "dinov2 small concat",
    "dinov2 small mean",
    "dinov3 small mean",
    "dinov2 small MIL",
    "SIFT",
    "AEGP Multiclass",
]


model_aliases = [
    "DINOv2s attn",
    "DINOv2s concat",
    "DINOv2s mean",
    "DINOv3s mean",
    "DINOv2s MIL",
    "SIFT",
    "AEGP\nMulticlass",
]


plot_switcher_accuracy_split_direct_labels(
    matrix=m,
    model_labels=model_labels,
    model_aliases=model_aliases,
    save_dir=".",
    prefix="state_estimation",
    jupyter_plot=False,
    ylim=(55, 100),
    min_label_gap=3.2,
    figsize_per_panel=(3.15, 3.25),
    base_fontsize=11,
    label_fontsize=11,
    title_fontsize=12,
)